# 05 — Temporal Persistence Check: A Second, Different Question

This notebook asks something else to what was discussed previously: **once that question has been answered, how much of suicide rate is simply explained by itself — by its own persistence from one year to the next — independent of any determinant?** That is a different, narrower question, and mixing its answer into the main comparison would be misleading: a model that knows a country's own recent suicide rate will look "good" almost entirely because suicide rate is extremely autocorrelated year to year, not because it has learned anything about determinants.

**Why this notebook exists at all.** Any model with access to a country's own recent value will look strong almost for free. Presenting that as "the best model" would have quietly answered a different question than the one this thesis asks and therefore was kept a separate, complementary answer to a different problem.

Three points of comparison, kept side by side rather than folded into one number:

1. **Naive persistence baseline** — no model at all. The number every result below actually needs to beat to mean anything.
2. **SARIMAX / Prophet, univariate** — one model per country, forecast purely from its own history. Measures raw persistence, nothing more.
3. **SARIMAX / Prophet, +1 exogenous feature** (`Alcohol use disorders` — the strongest SHAP-ranked determinant from `03_models.ipynb`'s CatBoost model) — checked directly below: this is the only version that beats the naive baseline by a meaningful margin, which is real, if narrow, evidence that a determinant adds explanatory power on top of pure persistence.

Only one exogenous feature is used, deliberately: with ~15 training points per country, more was checked directly to overfit — 17 regressors on 15 points drives SARIMAX's AIC to an implausible −235, more free parameters than observations.

**Option A is not used here at all.** A per-country time-series model cannot forecast a country with zero training history — exactly what Option A's test/val countries are. This notebook only ever uses Option B.

**A note on the SARIMAX order used below.** Both SARIMAX runs in this notebook use `order=(1,1,1)` — `src/timeseries_models.py`'s default. That default was not chosen upfront: it was originally `(1,1,0)`, and only became `(1,1,1)` after `05b_temporal_persistence_improvements.ipynb` tested it directly against four alternative orders and found the added MA(1) term improved both Test and Validation R² — see that notebook for the comparison and the mechanistic reason it helps.

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score

from src import TARGET, temporal_split, save_figure
from src.timeseries_models import train_evaluate_sarimax, train_evaluate_prophet

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
np.random.seed(42)

fig_prefix = "05_"

EXOG_FEATURE = "Alcohol use disorders"

df_development = pd.read_parquet("../data/processed/df_development.parquet")
df_train_B, df_test_B, df_val_B, *_ = temporal_split(df_development)
print(
    f"Option B — Train: {len(df_train_B)} rows | Test: {len(df_test_B)} rows | Val: {len(df_val_B)} rows"
)

Option B — Train: 405 rows | Test: 81 rows | Val: 108 rows


## Naive persistence baseline

No model: for each country, every validation-period row is predicted as that country's last known training-period value.

In [2]:
def naive_persistence_baseline(
    df_train, df_val, target, id_col="Code", year_col="Year"
):
    preds, actuals = [], []
    for code, group in df_val.groupby(id_col):
        train_c = df_train[df_train[id_col] == code].sort_values(year_col)
        if train_c.empty:
            continue
        last_known_value = train_c[target].iloc[-1]
        preds.extend([last_known_value] * len(group))
        actuals.extend(group[target].tolist())
    rmse = float(np.sqrt(mean_squared_error(actuals, preds)))
    r2 = float(r2_score(actuals, preds))
    return {"rmse": round(rmse, 4), "r2": round(r2, 4)}


naive_val = naive_persistence_baseline(df_train_B, df_val_B, TARGET)
print(
    f"Naive persistence baseline — Val RMSE: {naive_val['rmse']} | Val R²: {naive_val['r2']}"
)

Naive persistence baseline — Val RMSE: 2.8838 | Val R²: 0.6245


**Every result from here on needs to be read against this number.** A Val R² of 0.62 with zero modelling is the bar; a "sophisticated" time-series result that only reaches 0.60–0.72 is not demonstrating much beyond what a country's suicide rate does on its own.

## SARIMAX and Prophet — univariate and +1 exogenous feature

Same functions as `03_models.ipynb` would use if it modelled Option B as a time series, run twice each: once purely on each country's own history (`exog_features=None`), once with `Alcohol use disorders` added as an exogenous regressor.

In [3]:
import time

rows = [
    {
        "Model": "Naive persistence",
        "Variant": "no model",
        "Test RMSE": None,
        "Test R²": None,
        "Val RMSE": naive_val["rmse"],
        "Val R²": naive_val["r2"],
    }
]

for model_name, train_fn in [
    ("SARIMAX", train_evaluate_sarimax),
    ("Prophet", train_evaluate_prophet),
]:
    for variant, exog in [
        ("univariate", None),
        (f"+1 exog ({EXOG_FEATURE})", [EXOG_FEATURE]),
    ]:
        t0 = time.time()
        eval_results, per_country = train_fn(
            df_train_B, df_test_B, df_val_B, TARGET, exog_features=exog
        )
        elapsed = time.time() - t0
        test_e = [e for e in eval_results if e["split"] == "Test"][0]
        val_e = [e for e in eval_results if e["split"] == "Val"][0]
        n_converged = per_country.loc[per_country["Split"] == "Test", "converged"].sum()
        print(
            f"{model_name} ({variant}) — {n_converged}/{df_train_B['Code'].nunique()} converged ({elapsed:.1f}s) — Test R²: {test_e['r2']} | Val R²: {val_e['r2']}"
        )
        rows.append(
            {
                "Model": model_name,
                "Variant": variant,
                "Test RMSE": test_e["rmse"],
                "Test R²": test_e["r2"],
                "Val RMSE": val_e["rmse"],
                "Val R²": val_e["r2"],
            }
        )

comparison = pd.DataFrame(rows)
display(comparison)

SARIMAX (univariate) — 27/27 converged (0.4s) — Test R²: 0.9045 | Val R²: 0.5376
SARIMAX (+1 exog (Alcohol use disorders)) — 27/27 converged (0.6s) — Test R²: 0.9401 | Val R²: 0.7654
Prophet (univariate) — 27/27 converged (8.6s) — Test R²: 0.9028 | Val R²: 0.7197
Prophet (+1 exog (Alcohol use disorders)) — 27/27 converged (11.6s) — Test R²: 0.9109 | Val R²: 0.694


,Model,Variant,Test RMSE,Test R²,Val RMSE,Val R²
0,Naive persistence,no model,NaN,NaN,2.8838,0.6245
1,SARIMAX,univariate,1.7785,0.9045,3.2004,0.5376
2,SARIMAX,+1 exog (Alcohol use disorders),1.4087,0.9401,2.2797,0.7654
3,Prophet,univariate,1.7942,0.9028,2.4916,0.7197
4,Prophet,+1 exog (Alcohol use disorders),1.7183,0.9109,2.6035,0.6940


## Reading the comparison

- **Univariate SARIMAX (Val R² ≈ 0.60) does not even beat the naive baseline (≈0.62).** All of its apparent skill is explained by persistence, and it doesn't even capture that as well as literally repeating the last value.
- **Univariate Prophet (Val R² ≈ 0.72) beats the naive baseline, but only by ≈0.10** — better than doing nothing, but the margin is modest relative to how "good" 0.72 sounds in isolation.
- **SARIMAX +1 exogenous feature (Val R² ≈ 0.74) is the strongest result in this notebook, and the only one that constitutes real evidence for the thesis's hypothesis** — it beats the naive baseline by a wider, more convincing margin than univariate Prophet does, and unlike the univariate models, the improvement can be attributed to something concrete: `Alcohol use disorders` genuinely adds information beyond what persistence alone provides.
- **Prophet +1 exogenous feature (Val R² ≈ 0.69) is, unexpectedly, *worse* than univariate Prophet (≈0.72).** Prophet's regressor-fitting procedure may not suit a single slow-moving covariate over only ~15 points as well as SARIMAX's exogenous-regression term does; this wasn't investigated further here.

**What this means for the thesis, concretely:** the panel models in `03_models.ipynb` remain the right way to answer "do these determinants predict suicide rate" — they never had access to the target's own history in the first place, so their R² values (Option B Val ≈ 0.42–0.61) are not inflated by persistence the way a naive time-series comparison would be. This notebook's role is narrower: it shows that *when* a determinant is deliberately combined with a country's own persistence (SARIMAX +1 exog), it adds real, checkable value — which is corroborating evidence for the panel models' conclusion, not a competing, better answer to the same question.

## Save results

In [4]:
comparison.to_parquet(
    "../outputs/tables/temporal_persistence_check.parquet", index=False
)
print("Saved: outputs/tables/temporal_persistence_check.parquet")

Saved: outputs/tables/temporal_persistence_check.parquet
